# 03. Autograd and Backpropagation

## 학습 목표

1. `requires_grad=True`의 의미를 이해한다.
2. 연산 그래프와 자동 미분의 관계를 이해한다.
3. `backward()`와 `.grad`의 역할을 이해한다.
4. Chain Rule을 역전파와 연결한다.
5. Gradient 누적과 초기화를 이해한다.
6. `detach()`와 `torch.no_grad()`를 구분한다.
7. 간단한 선형 모델을 직접 학습한다.


In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


---

# 1. Autograd란?

Autograd는 PyTorch의 **자동 미분 시스템**이다.

Tensor에 `requires_grad=True`를 지정하면 PyTorch가 해당 Tensor가 참여한 연산을 추적한다. 최종 결과에서 `backward()`를 호출하면 각 Tensor에 대한 미분값, 즉 Gradient를 자동으로 계산한다.


In [2]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3

print("x:", x)
print("y:", y)
print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)

x: tensor(2., requires_grad=True)
y: tensor(8., grad_fn=<PowBackward0>)
x.requires_grad: True
y.requires_grad: True


수식은 다음과 같다.

$$
y=x^3
$$

$x=2$일 때:

$$
\frac{dy}{dx}=3x^2=12
$$


In [3]:
y.backward()

print("x.grad:", x.grad)

x.grad: tensor(12.)


`backward()`가 미분을 수행하고, 그 결과를 leaf Tensor인 `x`의 `.grad`에 저장했다.


---

# 2. 연산 그래프

다음 연산을 생각해 보자.

$$
a=x^2,\qquad y=3a+1
$$

PyTorch는 연산 관계를 그래프로 기록한다.


In [4]:
x = torch.tensor(2.0, requires_grad=True)
a = x**2
y = 3 * a + 1

print("x:", x)
print("a:", a)
print("y:", y)
print("a.grad_fn:", a.grad_fn)
print("y.grad_fn:", y.grad_fn)

x: tensor(2., requires_grad=True)
a: tensor(4., grad_fn=<PowBackward0>)
y: tensor(13., grad_fn=<AddBackward0>)
a.grad_fn: <PowBackward0 object at 0x7beabf0f7310>
y.grad_fn: <AddBackward0 object at 0x7beabf0f7310>


In [5]:
y.backward()

print("x.grad:", x.grad)

x.grad: tensor(12.)


전체 함수는 다음과 같다.

$$
y=3x^2+1
$$

따라서:

$$
\frac{dy}{dx}=6x
$$

$x=2$이면 Gradient는 12이다.


---

# 3. Chain Rule과 Backpropagation

여러 함수가 연결되어 있으면 Chain Rule을 사용한다.

$$
\frac{dy}{dx}
=
\frac{dy}{da}
\frac{da}{dx}
$$

위 예제에서는:

$$
\frac{dy}{da}=3,\qquad
\frac{da}{dx}=2x
$$

따라서:

$$
\frac{dy}{dx}=3\cdot2x=6x
$$

역전파는 출력에서 시작하여 이 미분들을 그래프의 반대 방향으로 전달한다.


## 연습 1

다음 함수의 $x=3$에서의 Gradient를 구하자.

$$
y=2x^3+5x
$$


In [6]:
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x**3 + 5 * x

y.backward()

print("y:", y)
print("x.grad:", x.grad)

y: tensor(69., grad_fn=<AddBackward0>)
x.grad: tensor(59.)


<details>
<summary>정답 확인</summary>

```python
y.backward()
print(x.grad)  # tensor(59.)
```

$$
\frac{dy}{dx}=6x^2+5
$$

$x=3$이면 $6\cdot9+5=59$이다.

</details>


---

# 4. 출력이 Scalar가 아닐 때

인수 없이 `backward()`를 호출하려면 일반적으로 최종 출력이 Scalar여야 한다.


In [7]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x**2

print("y:", y)
print("y.shape:", y.shape)

y: tensor([1., 4., 9.], grad_fn=<PowBackward0>)
y.shape: torch.Size([3])


`y`는 값이 3개이므로 `y.backward()`를 바로 호출할 수 없다. 합이나 평균을 사용하여 Scalar Loss를 만들 수 있다.


In [8]:
loss = y.sum()
loss.backward()

print("loss:", loss)
print("x.grad:", x.grad)

loss: tensor(14., grad_fn=<SumBackward0>)
x.grad: tensor([2., 4., 6.])


여기서는:

$$
loss=x_1^2+x_2^2+x_3^2
$$

따라서 Gradient는:

$$
[2x_1,2x_2,2x_3]=[2,4,6]
$$


## 외부 Gradient 전달

출력이 여러 값이면 각 출력의 기여도를 `gradient` 인수로 전달할 수도 있다.


In [9]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x**2

external_gradient = torch.tensor([1.0, 0.1, 0.01])
y.backward(gradient=external_gradient)

print("x.grad:", x.grad)

x.grad: tensor([2.0000, 0.4000, 0.0600])


입문 단계에서는 대부분 Loss를 하나의 Scalar로 만든 뒤 `loss.backward()`를 호출한다고 이해하면 충분하다.


---

# 5. Gradient는 누적된다

PyTorch는 `.grad`를 자동으로 덮어쓰지 않고 기존 값에 더한다.


In [10]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x**2
y1.backward()
print("첫 번째 backward:", x.grad)

y2 = x**2
y2.backward()
print("두 번째 backward:", x.grad)

첫 번째 backward: tensor(4.)
두 번째 backward: tensor(8.)


두 번째 역전파에서도 Gradient 4가 계산되지만, 기존 4에 더해져 8이 된다.


In [11]:
x.grad.zero_()

print("초기화 후:", x.grad)

초기화 후: tensor(0.)


모델 학습에서는 보통 다음을 사용한다.

```python
optimizer.zero_grad()
```

Optimizer가 관리하는 모든 Parameter의 Gradient를 초기화한다.


---

# 6. Leaf Tensor

사용자가 직접 만들고 `requires_grad=True`를 지정한 Tensor는 일반적으로 leaf Tensor이다. `.grad`는 기본적으로 leaf Tensor에 저장된다.


In [24]:
x = torch.tensor(2.0, requires_grad=True)
a = x * 3
y = a**2

print("x.is_leaf:", x.is_leaf)
print("a.is_leaf:", a.is_leaf)
print("y.is_leaf:", y.is_leaf)

y.backward()

print("x.grad:", x.grad)
print("a.grad:", a.grad)

x.is_leaf: True
a.is_leaf: False
y.is_leaf: False
x.grad: tensor(36.)
a.grad: None


/tmp/ipykernel_6443/1565889875.py:12: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print("a.grad:", a.grad)


중간 Tensor의 Gradient가 필요하다면 `retain_grad()`를 사용할 수 있다.


In [25]:
x = torch.tensor(2.0, requires_grad=True)
a = x * 3
a.retain_grad()

y = a**2
y.backward()

print("x.grad:", x.grad)
print("a.grad:", a.grad)

x.grad: tensor(36.)
a.grad: tensor(12.)


---

# 7. `detach()`

`detach()`는 Tensor를 현재 연산 그래프에서 분리한다.


In [26]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2
z = y.detach()

print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad)
print("y:", y)
print("z:", z)

y.requires_grad: True
z.requires_grad: False
y: tensor(4., grad_fn=<PowBackward0>)
z: tensor(4.)


`z`는 같은 값을 가지지만, `z` 이후의 연산은 `x`까지 Gradient를 전달하지 않는다.


---

# 8. `torch.no_grad()`

코드 블록 전체에서 Gradient 추적을 중단한다.


In [27]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x**2 + 1

print("y:", y)
print("y.requires_grad:", y.requires_grad)

y: tensor(5.)
y.requires_grad: False


모델 추론에서는 보통 다음처럼 사용한다.

```python
model.eval()

with torch.no_grad():
    prediction = model(x)
```

- `detach()`: 특정 Tensor를 그래프에서 분리
- `no_grad()`: 코드 블록 전체에서 연산 추적 중단


---

# 9. In-place 연산 주의

이름 뒤에 `_`가 붙은 함수는 원본 Tensor를 직접 수정하는 경우가 많다.

```python
x.add_(1)
x.zero_()
```

Autograd가 추적 중인 값을 함부로 수정하면 역전파에 필요한 정보가 사라질 수 있다.


In [28]:
x = torch.tensor(2.0, requires_grad=True)

# x.add_(1)  # leaf Tensor 직접 수정: 오류

with torch.no_grad():
    x.add_(1)

print(x)

tensor(3., requires_grad=True)


---

# 10. 선형 회귀를 수동으로 학습하기

다음 관계를 학습한다.

$$
y=2x+1
$$

모델:

$$
\hat{y}=wx+b
$$

Loss:

$$
L=\frac{1}{N}\sum_i(\hat y_i-y_i)^2
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_true = torch.tensor([3.0, 5.0, 7.0, 9.0])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.05

for epoch in range(100):
    y_pred = w * x + b
    loss = ((y_pred - y_true) ** 2).mean()

    loss.backward()

    # 업데이트 연산까지 추적하면 Parameter 수정 과정 자체가 새로운 연산 그래프에 포함되어 그래프가 불필요하게 계속 이어질 수 있으므로
    # 업데이트 연산에는 gradient 추적을 하지 않는다. (어찌 보면 당연한 얘기이네..)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # Parameter를 업데이트하고 나서는 gradient누적을 막기 위해 0으로 초기화한다.
    w.grad.zero_()
    b.grad.zero_()

    if epoch % 20 == 0:
        print(
            f"epoch={epoch:3d}, "
            f"loss={loss.item():.6f}, "
            f"w={w.item():.4f}, "
            f"b={b.item():.4f}"
        )

print("최종 w:", w.item())
print("최종 b:", b.item())

epoch=  0, loss=41.000000, w=1.7500, b=0.6000
epoch= 20, loss=0.007504, w=2.0710, b=0.7912
epoch= 40, loss=0.004105, w=2.0525, b=0.8456
epoch= 60, loss=0.002245, w=2.0388, b=0.8858
epoch= 80, loss=0.001228, w=2.0287, b=0.9155
최종 w: 2.021571397781372
최종 b: 0.9365772008895874


학습 순서:

1. Forward
2. Loss 계산
3. `backward()`
4. Parameter 업데이트
5. Gradient 초기화


---

# 11. 종합 연습문제

## 문제 1

$$
y=(x^2+1)^3
$$

$x=2$일 때 Gradient를 구하자.


In [30]:
x = torch.tensor(2.0, requires_grad=True)
y = (x**2 + 1) ** 3

y.backward()

print("y:", y)
print("x.grad:", x.grad)

y: tensor(125., grad_fn=<PowBackward0>)
x.grad: tensor(300.)


## 문제 2

두 번째 역전파 전에 Gradient를 초기화하자.


In [32]:
x = torch.tensor(3.0, requires_grad=True)

y1 = x**2
y1.backward()

x.grad.zero_()

y2 = 2 * x
y2.backward()

print("x.grad:", x.grad)

x.grad: tensor(2.)


## 문제 3

`torch.no_grad()`를 사용하여 Gradient 추적 없이 $x^2+1$을 계산하자.


In [33]:
x = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    y = x**2 + 1

print("y:", y)
print("requires_grad:", y.requires_grad if y is not None else None)

y: tensor(26.)
requires_grad: False


---

# 12. 종합 연습문제 정답


In [21]:
# 문제 1
x = torch.tensor(2.0, requires_grad=True)
y = (x**2 + 1) ** 3
y.backward()

print("y:", y)
print("x.grad:", x.grad)

y: tensor(125., grad_fn=<PowBackward0>)
x.grad: tensor(300.)


$$
\frac{dy}{dx}=3(x^2+1)^2\cdot2x
$$

$x=2$이면 $3\cdot25\cdot4=300$이다.


In [22]:
# 문제 2
x = torch.tensor(3.0, requires_grad=True)

y1 = x**2
y1.backward()

x.grad.zero_()

y2 = 2 * x
y2.backward()

print("x.grad:", x.grad)

x.grad: tensor(2.)


In [23]:
# 문제 3
x = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    y = x**2 + 1

print("y:", y)
print("requires_grad:", y.requires_grad)

y: tensor(26.)
requires_grad: False


---

# 13. 핵심 정리

- `requires_grad=True`: 연산 추적
- `backward()`: 역전파
- `.grad`: Gradient 저장
- Gradient는 기본적으로 누적
- `zero_()` 또는 `optimizer.zero_grad()`: Gradient 초기화
- `detach()`: 특정 Tensor를 그래프에서 분리
- `torch.no_grad()`: 코드 블록 전체의 추적 중단
- 학습 순서: Forward → Loss → Backward → Update → Zero Gradient
